# [claude_v2] 실험2: 검증 fold early stopping 제거 + 전체 재학습

민정 v10(`models/민정/model_민정_v10.ipynb`) 복사본 위에 실험 하나만 켠 버전.

**이번 버전 변경점**
- 반복 수를 검증 fold가 아니라 **학습 fold 내부 15% 분할**로 정한 뒤 학습 fold 전체로 재학습 -> OOF가 정직해짐.
- test 예측은 fold 모델 평균(각 80% 데이터) 대신 **train 100%로 재학습**한 모델로.

**모든 claude 버전 공통 (비교 조건 통일)**
- 1번 셀 끝의 공통 설정에서 이 버전의 플래그만 켬 (`VERSION_CFG`). 나머지는 민정 v10과 동일.
- 누수 점검: 프로필 해시 번호표를 train으로만 만들고 test는 조회만 (v10은 train+test를 합쳐 factorize).
- 잡음 스트레스 평가(`NOISE_EVAL`): 검증 행에 가짜 변이를 넣어 같은 fold 모델로 예측. 학습에는 영향 없음.
- 비교 지표: `oof_final_nested` = 중복 규칙(fold 표)·후처리(다른 fold로 맞춤)까지 포함한 제출 파이프라인의 OOF macroF1.
- 결과는 `results/claude_<버전>.json`, 제출 후보는 `submission_claude_<버전>_norm_dup.csv`.


# 암종 분류 v6 (v3 + 쌍둥이 규칙 + 표기 통일): 이진 행렬 + 파생변수 + 인프레임 핫스팟 + 동반변이 콤보 + XGB/LR 앙상블 (5-fold CV)

**v2 대비 변경점**
- 인프레임 변이(`EGFR E746_A750del` 등)를 핫스팟 토큰으로 추가 (`gene|inframe`, `gene|위치_inframe`)
- 가이드 기반 동반변이 콤보 피처 (`cb_*`), TMB 기준 5 / 30
- **EM 사전확률 보정 제거** (macroF1을 0.46 -> 0.21로 망가뜨림). COAD 확률 보정은 검토 후 이득이 기준 미만(미적용)이고 test 분포 정보가 필요해서 **삭제**
- 실행 순서 버그 수정: 예전 prior 셀이 CV보다 앞에 있어 `oof_ens`가 정의되기 전에 쓰였음 -> 앙상블 이후로 이동
- CV를 함수(`run_cv`)로 묶고, 콤보 ON/OFF 비교(ablation) 셀 추가

**패키지**: `pip install pandas numpy scikit-learn xgboost`


**v5 변경점**: 쌍둥이(형제 라벨) 규칙 추가 (5-2절). KIRC↔KIPAN, LGG↔GBMLGG 는 같은 환자가 두 코호트에 중복 등록된 경우가 있어, train에 짝 라벨이 있는 프로필이 test에 나오면 짝 암종으로 교정한다. (최소빈도는 v3 그대로 5)

**v6 주요 변경점**: 변이 표기 정규화 (`NORMALIZE_TOKENS`, `canon`)
- HGVS 규약상 같은 변이가 `R1450X` / `R1450*` / `p.Arg1450Ter` 등으로 달리 적힐 수 있어, 표기와 무관하게 같은 변이가 같은 피처가 되도록 정규화하고 한 칸 안의 중복 표기를 제거한다.
- 모든 연산은 토큰/행 하나만 보는 독립 연산이며 train/test에 동일하게 적용된다. test 통계는 사용하지 않는다.
- 단, train에는 구식 표기가 0건이라 train만으로 효과를 검증할 수 없다. 운영진 확인 전까지는 `NORMALIZE_TOKENS=False` 제출본(`nonorm`)을 함께 준비한다.
- **v7 보강(canon)**: 괄호 `p.(Arg1450Ter)`, 대문자 `P.`, 소문자 `x`/`ter`, 대소문자 섞인 3글자 표기도 같은 변이로 통일. train에는 해당 표기가 0건이라 기존 결과는 변하지 않는다.

**v8 변경점 (TwinRuleRefine.md 반영)**: 후처리 규칙으로 점수를 짜내는 대신 데이터 정의와 분할 단계에서 푼다.
- **그룹 분할 CV** (`GROUP_CV=True`): 변이 프로필이 같은 행(쌍둥이·중복)은 같은 fold에 묶는다. 같은 프로필이 학습/검증에 갈라져 OOF가 부풀려지는 것을 막는다. (`StratifiedGroupKFold`, 그룹 = 프로필 해시)
- **프로필 중복·라벨 충돌 진단 셀** 추가 (4-0절): 데이터 큐레이션의 근거.
- **쌍둥이 규칙 기본 OFF** (`USE_TWIN=False`): 분석가가 라벨 구조를 이용해 예측을 덮어쓰는 후처리라 확장성·규정 리스크가 크다. 진단용으로만 남겼고, 켜려면 플래그와 시뮬 이득 조건을 모두 만족해야 한다.
- (선택) `PROFILE_WEIGHT`: 같은 프로필 행이 학습을 지배하지 않도록 1/그룹크기 가중치. 기본 OFF, 효과는 CV로 확인 필요.
- 주의: 그룹 분할로 바뀌면 OOF 점수는 이전(무작위 분할)보다 낮아질 수 있다. 이전 점수가 중복 덕분에 부풀려져 있었다는 뜻이지 모델이 나빠진 것이 아니다.

**v9 변경점 (희소 변이 샘플 대응)**: 변이가 거의 없는 샘플이 다른 암종으로 오인되는 문제를 다룬다. 모두 CFG 플래그로 켜고 끌 수 있고, 4-2절 ablation으로 효과를 확인한 뒤 쓴다.
- `USE_SPARSE_META`: 변이 유전자 0개 / 2개 이하 / `TMB_LOW` 이하 희소성 지표 피처 (`sp_*`)
- `USE_KNN`: Jaccard 유사도 기반 KNN 피처 (`knn_p_*` = 가까운 train 환자들의 암종 분포, `knn_max_sim`, `knn_mean_sim`). **변이 프로필이 완전히 같은 행(자기 자신·쌍둥이·중복)은 이웃에서 제외**해서, 쌍둥이 라벨을 몰래 학습하는 우회로가 되지 않게 했다. 이웃은 항상 그 fold의 train 행에서만 찾는다 (valid/test도 동일).
- `SPARSE_WEIGHT`: 변이가 2개 이하인 학습 행의 가중치 배수 (1.0=사용 안 함)

**v10 변경점 (점수 + 재현성 + 현업성)**
- **쌍둥이 규칙 -> 학습형 중복 규칙(`USE_DUP_RULE`)**: 암종 이름을 코드에 박아 둔 `TWIN_SWAP` 대신, train에서 "같은 프로필을 가진 *다른* train 행들의 라벨 묶음 -> 내 라벨" 표를 leave-one-out으로 추정해 쓴다. 표본 수(`DUP_MIN_SUPPORT`)와 확률(`DUP_MIN_PROB`)이 기준 미만이면 규칙이 스스로 꺼지고, 어떤 암종 쌍에도 일반화된다. OOF 시뮬 이득이 `BOOST_MIN_GAIN` 미만이면 적용하지 않는다.
- **macroF1 클래스 보정(`USE_CLASS_SCALE`)**: 클래스별 확률 배율을 OOF로 맞춘다. 중첩 검증(다른 fold에서 맞추고 남은 fold로 평가)에서 이득이 확인될 때만 적용한다. test 분포 정보는 쓰지 않는다.
- **정직한 점수 보고(`HONEST_REPORT`)**: 주 CV는 무작위 분할(대회 조건과 같고 이전 점수와 비교 가능), 추가로 그룹 분할 점수를 따로 출력한다. 현업 기준 점수는 그룹 분할 쪽이다.
- **재현성 기록**: 실행 설정, 패키지 버전, 데이터 파일 해시, 점수, 적용된 후처리를 `run_manifest_*.json`으로 저장한다.
- **제출 파일 2종**: 규칙 없는 `_core`(현업형)와 규칙 적용 `_dup`을 함께 만든다. 어느 쪽을 낼지는 대회 규정 확인 후 선택.
- v9의 희소성/KNN 피처는 실데이터 검증 전이라 기본 OFF로 되돌렸다 (4-2절 ablation으로 확인 후 켜기).


In [1]:
import os, re, warnings, time
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("DATA_DIR", "../../data"),
    SEED=42,
    N_FOLDS=int(os.environ.get("N_FOLDS", 5)),
    MIN_GENE_FREQ=5,        # train에서 변이 샘플이 이 값 미만인 유전자는 제외
    MIN_HOTSPOT_FREQ=5,     # gene|position 토큰 최소 빈도
    MIN_AA_FREQ=20,         # 아미노산 치환쌍 최소 빈도
    USE_AA_SUBS=True,       # 아미노산 치환 패턴 피처 on/off (CV로 효과 확인)
    USE_PATHWAYS=True,      # 경로 집계 피처 on/off
    USE_HOTSPOTS=True,      # gene|position, gene|trunc 토큰 on/off
    BALANCE=True,           # 클래스 불균형 보정 (sample weight / class_weight)
    XGB_N_EST=int(os.environ.get("XGB_N_EST", 2000)),
    LR_C=0.1,
    USE_COMBOS=True,        # 동반 변이 콤보 피처 on/off
    TMB_LOW=5,              # 극저변이 기준 (THCA/LAML 등)
    TMB_HIGH=30,            # 고변이 기준 (SKCM/UCEC/STES 등)
    BOOST_MIN_GAIN=0.003,   # 쌍둥이 규칙 등 후처리가 OOF macroF1을 이만큼 못 올리면 적용 안 함
    NORMALIZE_TOKENS=True,  # 변이 표기 통일(HGVS 구/신 표기) + 한 칸 안 중복 제거. 운영진 확인 전에는 False 제출본도 준비
    USE_DUP_RULE=True,      # 학습형 중복 규칙: 같은 프로필의 train 라벨 묶음 -> 라벨 표(LOO)로 덮어쓰기. 근거 부족/이득 없으면 자동 OFF
    DUP_MAX_CTX=3,          # 같은 프로필 행이 이 개수 이하인 맥락만 사용 (큰 중복 군집은 제외)
    DUP_MIN_SUPPORT=20,     # 표의 한 칸이 이 표본 수 이상이어야 사용
    DUP_MIN_PROB=0.90,      # 표의 한 칸에서 최다 라벨 비율이 이 이상이어야 사용
    USE_CLASS_SCALE=True,   # macroF1용 클래스별 확률 배율 (중첩 검증 이득이 BOOST_MIN_GAIN 이상일 때만 적용)
    SCALE_ITERS=3,          # 배율 좌표 탐색 반복 횟수
    HONEST_REPORT=True,     # 주 CV가 무작위 분할일 때, 그룹 분할 점수를 추가로 계산해 출력 (시간 약 2배)
    GROUP_CV=False,         # True면 주 CV를 그룹 분할로 (중복 규칙 시뮬 불가). 기본 False: 대회 조건과 같은 무작위 분할 + HONEST_REPORT로 그룹 점수 병기
    PROFILE_WEIGHT=False,   # 같은 프로필 행에 1/그룹크기 가중치 (중복 행이 학습을 지배하지 않게). CV로 확인 후 사용
    USE_SPARSE_META=False,   # 희소성 메타 피처(변이 유전자 0/<=2/<=TMB_LOW 지표)
    USE_KNN=False,          # Jaccard KNN 피처 (같은 프로필 행은 이웃에서 제외)
    KNN_K=15,               # 이웃 수
    SPARSE_WEIGHT=1.0,      # 변이 2개 이하 학습 행의 가중치 배수 (1.0=off, 예: 0.5)
    RUN_ABLATION2=False,    # True면 희소성/KNN 피처 ON/OFF 비교 (4-2절, CV 여러 번)
    RUN_ABLATION=False,     # True면 콤보 ON/OFF CV 비교 (시간 2배)
)

VERSION_CFG = {'INNER_ES': True, 'FULL_REFIT': True, 'VERSION': 'claude_v2'}

# ================= claude 실험 공통 설정 (민정 v10 위에 추가) =================
import json
CFG.update(
    DATA_DIR=os.environ.get("DATA_DIR", "../../../data"),   # models/혜진/claude 기준
    HONEST_REPORT=False,     # 실험 비교에서는 시간 절약을 위해 끔 (그룹 분할 점수는 최종본에서만)
    VERSION="claude_v0", OUT_DIR=".",
    NOISE_EVAL=True,         # 잡음 스트레스 평가: 검증 행에 가짜 변이(평균 NOISE_MEAN개)를 넣고 같은 fold 모델로 예측
    NOISE_MEAN=20,
    AUG_NOISE=False,         # [v1] 학습 fold 행의 잡음 복사본을 추가 학습 (가중치 AUG_WEIGHT)
    AUG_MEAN=10, AUG_WEIGHT=0.5,
    GENE_MODE="all",         # [v1] all=기존 g_(silent 포함) / nonsilent=silent 제외 g_ / both=둘 다
    INNER_ES=False,          # [v2] early stopping을 검증 fold가 아니라 학습 fold 안의 내부 분할(15%)로
    REFIT_MULT=1.15,         # [v2] 내부 분할로 찾은 반복 수 x 이 값으로 학습 fold 전체 재학습
    FULL_REFIT=False,        # [v2] test 예측을 train 전체 재학습 모델로 (fold 평균 대신)
    FULL_MULT=1.1,           # [v2] 전체 재학습 반복 수 = fold 평균 반복 수 x 이 값
    XGB_LR=0.08, XGB_SEEDS=[42],   # [v4] 학습률 / seed 평균
    USE_LGB=False, LGB_LR=0.05,    # [v4] LightGBM 추가
    LR_CS=[0.1],                   # [v4] LR C 후보 (여러 개면 OOF로 선택)
    USE_PAIR_BIAS=False,           # [v3] 중복 표에서 학습한 형제 암종 쌍에만 확률 배율 (중첩 검증)
)
CFG.update(VERSION_CFG)
for _k in list(CFG):                 # 환경변수 CFG_<키>=<json> 으로 덮어쓰기 (ablation 실행용)
    _v = os.environ.get("CFG_" + _k)
    if _v is not None:
        CFG[_k] = json.loads(_v)
CFG["XGB_SEEDS"] = list(CFG["XGB_SEEDS"])
print({k: CFG[k] for k in ("VERSION", "AUG_NOISE", "GENE_MODE", "INNER_ES", "FULL_REFIT", "XGB_LR", "XGB_SEEDS",
                          "USE_LGB", "LR_CS", "USE_PAIR_BIAS", "PROFILE_WEIGHT")})


{'VERSION': 'claude_v2', 'AUG_NOISE': False, 'GENE_MODE': 'all', 'INNER_ES': True, 'FULL_REFIT': True, 'XGB_LR': 0.08, 'XGB_SEEDS': [42], 'USE_LGB': False, 'LR_CS': [0.1], 'USE_PAIR_BIAS': False, 'PROFILE_WEIGHT': False}


## 1. 데이터 로드

In [2]:
D = CFG["DATA_DIR"]
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")
sub = pd.read_csv(f"{D}/sample_submission.csv")

assert (test["ID"].values == sub["ID"].values).all(), "test와 sample_submission의 ID 순서가 다릅니다"
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
assert gene_cols == [c for c in test.columns if c != "ID"], "train/test 유전자 컬럼이 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])
n_class = len(le.classes_)
print("train", train.shape, "classes", n_class)
print(train["SUBCLASS"].value_counts())

train (6201, 4386) classes 26
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


In [3]:
train.head(5)

## EDA

In [4]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 로드
train = pd.read_csv(f"{D}/train.csv").fillna("WT")

print("="*50)
print("1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인")
print("="*50)
print(f"Train 크기: {train.shape}")

# 암종(SUBCLASS) 개수 및 분포 확인
subclass_counts = train["SUBCLASS"].value_counts()
print(f"총 암종(Subclass) 개수: {len(subclass_counts)}개")
print("\n[상위 5개 암종 분포]")
print(subclass_counts.head())

# 유전자 컬럼 분리
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
print(f"\n유전자(Feature) 컬럼 총 개수: {len(gene_cols)}개")


print("\n" + "="*50)
print("2. 샘플당 변이 유전자 수 분포 (TMB)")
print("="*50)
# WT가 아닌 유전자의 개수를 행별로 카운트
mut_matrix = (train[gene_cols] != "WT")
mut_counts_per_sample = mut_matrix.sum(axis=1)
print(mut_counts_per_sample.describe().round(2))


print("\n" + "="*50)
print("3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)")
print("="*50)
# WT가 아닌 셀들만 추출
all_vals = train[gene_cols].values.ravel()
non_wt_vals = all_vals[all_vals != "WT"]

non_wt_series = pd.Series(non_wt_vals)
# 쉼표(,), 콜론(:), 세미콜론(;), 파이프(|), 공백 등이 포함된 비율 확인
sep_pattern = r"[,:;|\s]"
has_sep = non_wt_series.str.contains(sep_pattern, regex=True).mean()
print(f"Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: {has_sep:.4%}")


print("\n" + "="*50)
print("4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)")
print("="*50)
# 숫자를 'N'으로 치환하여 대표적인 변이 표기 패턴 분석
pattern_samples = non_wt_series.str.replace(r"\d+", "N", regex=True)
print(pattern_samples.value_counts().head(15))


print("\n" + "="*50)
print("5. 유전자별 변이 빈도 (희소성 확인)")
print("="*50)
gene_mut_freq = mut_matrix.sum(axis=0)
print(f"변이가 단 1개라도 발생한 유전자 수: {(gene_mut_freq > 0).sum()} / {len(gene_cols)}")
print(f"대다수 유전자의 변이 빈도 요약 (상위/하위):")
print(gene_mut_freq.describe().round(2))

1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인
Train 크기: (6201, 4386)
총 암종(Subclass) 개수: 26개

[상위 5개 암종 분포]
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
Name: count, dtype: int64

유전자(Feature) 컬럼 총 개수: 4384개

2. 샘플당 변이 유전자 수 분포 (TMB)


count    6201.00
mean       35.30
std       100.78
min         0.00
25%         7.00
50%        14.00
75%        27.00
max      2393.00
dtype: float64

3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)


Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: 10.0625%

4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)
LNL    7409
ENK    6790
SNS    4517
RNH    4441
RNC    4141
RNQ    4127
PNP    4037
ANA    3935
TNT    3841
ANT    3712
DNN    3685
INI    3680
ANV    3652
GNG    3581
PNS    3536
Name: count, dtype: int64

5. 유전자별 변이 빈도 (희소성 확인)
변이가 단 1개라도 발생한 유전자 수: 4230 / 4384
대다수 유전자의 변이 빈도 요약 (상위/하위):
count    4384.00
mean       49.93
std        57.64
min         0.00
25%        20.00
50%        36.00
75%        61.00
max      1770.00
dtype: float64


## 2. 변이 표기 형식 확인 (파싱 규칙이 맞는지 눈으로 검증)

In [5]:
sample = pd.Series(train[gene_cols].iloc[:1000].to_numpy(dtype=object).ravel())
nonwt = sample[sample != "WT"]
print(f"non-WT 비율: {len(nonwt)/len(sample):.4%}")
print("한 셀에 구분자(, : ; | 공백)가 있는 비율:", nonwt.str.contains(r"[,:;|\s]").mean().round(4))
print(nonwt.str.replace(r"\d+", "N", regex=True).value_counts().head(25))

non-WT 비율: 0.8337%
한 셀에 구분자(, : ; | 공백)가 있는 비율: 0.0873
LNL    1236
ENK    1125
SNS     777
RNH     752
ANA     738
RNC     724
PNP     711
RNQ     696
TNT     665
ANT     642
GNG     636
ANV     629
FNF     614
PNL     610
DNN     609
PNS     593
INI     561
VNV     555
RNR     492
RNW     483
SNF     437
RN*     410
EN*     388
GNR     341
KNN     328
Name: count, dtype: int64


## 3. 피처 생성

- `g_*` : 유전자별 변이 여부 (이진)
- TMB 계열 : 변이 유전자 수, log, 이벤트 수
- 변이 유형 : missense / silent / nonsense / frameshift / splice / inframe 개수와 비율
- `hs_*` : 핫스팟(`유전자|원래아미노산+위치`) 및 truncating(`유전자|trunc`) 이진
- `aa_*` : 아미노산 치환쌍 비율
- `pw_*` : 경로별 변이 유전자 수 (아래 `PATHWAYS` 사전은 초안이니 자유롭게 수정)

In [6]:
def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [7]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
TYPES = ["missense", "silent", "nonsense", "frameshift", "splice", "inframe", "other"]
TYPE_IDX = {t: i for i, t in enumerate(TYPES)}
SEP = re.compile(r"[,:;|\s]+")
AA3 = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
       "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*"}

_A3 = "|".join(AA3)   # Ala|Arg|...|Ter
# 단순 치환형(예: Arg1450Ter, r1450x, ARG1450TER, p.(Arg1450Ter))은 통째로 매칭해서 대소문자까지 한 번에 통일
_SIMPLE = re.compile(rf"^({_A3}|[A-Z])(\d+)({_A3}|[A-Z]|\*)$", re.I)

def _aa1(x, last=False):
    """아미노산 표기 하나 -> 1글자. 마지막 자리의 X/x/Ter는 종결 코돈(*)으로 읽는다."""
    if x == "*":
        return x
    if len(x) == 3:
        return AA3.get(x.capitalize(), x)
    x = x.upper()
    return "*" if (last and x == "X") else x

def canon(tok):
    """변이 표기 1개를 한 가지 모양으로 통일한다 (그 토큰만 보고 바꿈 = 행 독립, train/test 동일 적용).
    p.Arg1450Ter / p.(Arg1450Ter) / R1450X / r1450x / R1450* -> R1450*   (옛 HGVS의 X도 종결 코돈으로 읽는 기존 동작 유지)
    v7 보강: 괄호(HGVS 예측 표기), 대문자 P., 소문자 x/ter, 대소문자 섞인 3글자 표기도 같은 변이로 통일."""
    t = tok.strip()
    t = re.sub(r"^\(?[pP]\.", "", t)      # p. / P. / (p. 접두 제거
    t = t.strip("()")                       # p.(Arg1450Ter) 같은 괄호 제거
    m = _SIMPLE.match(t)
    if m:
        a, pos, b = m.groups()
        a1, b1 = _aa1(a), _aa1(b, last=True)
        if a1 != "*":                       # 종결 코돈이 시작 자리에 오는 표기는 건드리지 않는다
            return f"{a1}{pos}{b1}"
    t = re.sub(r"[A-Z][a-z]{2}", lambda m: AA3.get(m.group(), m.group()), t)   # 3글자 아미노산 -> 1글자 (fs/del 등 복합 표기용)
    t = re.sub(r"^([A-Z])(\d+)[Xx]$", r"\1\2*", t)                             # R1450X -> R1450*
    return t

def cell_tokens(cell):
    """셀 하나 -> 변이 토큰 목록. NORMALIZE_TOKENS면 표기를 통일하고 같은 변이의 중복 표기를 하나로 합친다."""
    toks = [t for t in SEP.split(str(cell)) if t and t != "WT"]
    if CFG["NORMALIZE_TOKENS"]:
        toks = list(dict.fromkeys(canon(t) for t in toks))
    return toks
RE_MIS = re.compile(r"^(?:p\.)?([A-Z])(\d+)([A-Z])$")
RE_NON = re.compile(r"^(?:p\.)?[A-Z]\d+(\*|X|Ter)$")

def parse_token(tok):
    """변이 표기 1개 -> (유형, 원래aa, 위치, 바뀐aa). 표기가 다르면 여기 규칙을 수정하세요."""
    low = tok.lower()
    m = re.search(r"\d+", tok)
    pos = m.group() if m else ""
    if "fs" in low or "frameshift" in low:
        return "frameshift", "", pos, ""
    if "splice" in low:
        return "splice", "", pos, ""
    if "del" in low or "ins" in low or "dup" in low:
        return "inframe", "", pos, ""
    if RE_NON.match(tok):
        return "nonsense", "", pos, ""
    m = RE_MIS.match(tok)
    if m:
        fa, p, ta = m.groups()
        return ("silent" if fa == ta else "missense"), fa, p, ta
    return "other", "", pos, ""

def extract(df):
    arr = df[gene_cols].to_numpy(dtype=object)
    mut = arr != "WT"
    ns = np.zeros_like(mut)                 # silent가 아닌 변이가 있는 칸 (v1 GENE_MODE용)
    n = len(df)
    type_cnt = np.zeros((n, len(TYPES)), np.float32)
    aa_cnt = np.zeros((n, 400), np.float32)
    tok_rows, tok_names = [], []
    rr, cc = np.nonzero(mut)
    for r, c in zip(rr, cc):
        gene = gene_cols[c]
        for tok in cell_tokens(arr[r, c]):
            typ, fa, pos, ta = parse_token(tok)
            if typ != "silent": ns[r, c] = True
            type_cnt[r, TYPE_IDX[typ]] += 1
            if typ == "missense":
                tok_rows.append(r); tok_names.append(f"{gene}|{fa}{pos}")
                if fa in AA_IDX and ta in AA_IDX:
                    aa_cnt[r, AA_IDX[fa] * 20 + AA_IDX[ta]] += 1
            elif typ in ("nonsense", "frameshift", "splice"):
                tok_rows.append(r); tok_names.append(f"{gene}|trunc")
            elif typ == "inframe":                       # ← 추가
                tok_rows.append(r); tok_names.append(f"{gene}|inframe")        # 유전자 단위
                tok_rows.append(r); tok_names.append(f"{gene}|{pos}_inframe")  # 위치 단위 (EGFR 746_inframe 등)
    return dict(mut=mut, ns=ns, type_cnt=type_cnt, aa_cnt=aa_cnt,
                tok_rows=np.array(tok_rows, dtype=np.int64), tok_names=tok_names)

ex_tr, ex_te = extract(train), extract(test)
print("파싱 완료. 유형별 총 개수(train):", dict(zip(TYPES, ex_tr["type_cnt"].sum(0).astype(int))))
# ---- 표기 통일 동작 확인 ----
assert canon("R1450X") == "R1450*" == canon("p.Arg1450Ter") == canon("p.R1450*")
assert canon("E746_A750del") == "E746_A750del" and canon("S221Kfs") == "S221Kfs"
assert parse_token(canon("R1450X"))[0] == "nonsense"
# v7 보강 케이스
assert canon("p.(Arg1450Ter)") == canon("R1450x") == canon("ARG1450TER") == canon("P.Arg1450X") == "R1450*"
assert canon("p.Val600Glu") == canon("(V600E)") == canon("v600e") == "V600E"
assert canon("p.Arg97ProfsTer23") == "R97Pfs*23"
assert canon("X123R") == "X123R" and canon("Ter123Arg") == "*123R"
assert canon("p.Arg1450Xaa") == "R1450Xaa"      # 미지정 아미노산은 종결로 읽지 않는다
_save = CFG["NORMALIZE_TOKENS"]; CFG["NORMALIZE_TOKENS"] = True
assert cell_tokens("R1450X R1450* p.Arg1450Ter") == ["R1450*"]          # 같은 변이의 여러 표기 -> 1개
assert cell_tokens("R175H, V600E") == ["R175H", "V600E"] and cell_tokens("WT") == []
CFG["NORMALIZE_TOKENS"] = _save


파싱 완료. 유형별 총 개수(train): {'missense': np.int64(161051), 'silent': np.int64(64844), 'nonsense': np.int64(13080), 'frameshift': np.int64(9767), 'splice': np.int64(0), 'inframe': np.int64(3), 'other': np.int64(319)}


### 표기 점검 (train 전용)
train 데이터의 변이 표기 현황을 확인한다. HGVS 표기 규약상 같은 변이가 여러 방식(`R1450X`, `R1450*`, `p.Arg1450Ter`)으로 적힐 수 있어, 입력 표기가 달라도 같은 변이로 취급하도록 정규화(`canon`)를 둔다.
정규화는 토큰 하나만 보고 바꾸는 행 독립 연산이며 train/test에 동일하게 적용된다.

**train 점검 결과**: 구식 종결 표기(`숫자+X`), `p.` 접두, 3글자 아미노산 표기는 train에 **0건**이다. 즉 이 정규화는 train 데이터로 필요성이 확인된 것이 아니라 HGVS 규약에 따른 방어적 처리이며, 적용 여부는 대회 운영진 확인 결과에 따른다 (`NORMALIZE_TOKENS` 플래그로 on/off, 제출 파일명에 norm/nonorm 표시).


In [8]:
# train에서 표기 혼재 여부를 확인한다 (test는 사용하지 않음)
def tokens_per_cell(df, normalize):
    """non-WT 셀 하나당 평균 변이 토큰 수 (normalize=True면 canon 적용 + 중복 제거)."""
    vals = df[gene_cols].to_numpy(dtype=object).ravel()
    vals = vals[vals != "WT"]
    n = [len(dict.fromkeys(canon(t) for t in SEP.split(str(v)) if t and t != "WT")) if normalize
         else len([t for t in SEP.split(str(v)) if t and t != "WT"]) for v in vals]
    return float(np.mean(n)) if n else 0.0

_vals = pd.Series(train[gene_cols].to_numpy(dtype=object).ravel())
_vals = _vals[_vals != "WT"]
_toks = pd.Series([t for v in _vals for t in SEP.split(str(v)) if t and t != "WT"])

x_end = _toks[_toks.str.contains(r"^(?:p\.)?[A-Z]\d+X$")]
print(f"train 전체 토큰 {len(_toks):,}개 중 '숫자+X' 종결 표기: {len(x_end)}개")
print(x_end.value_counts().head(10))
print("p. 접두 표기:", int(_toks.str.startswith("p.").sum()),
      "| 3글자 아미노산 표기:", int(_toks.str.contains(r"[A-Z][a-z]{2}\d|\d[A-Z][a-z]{2}").sum()))
print(f"train 칸당 변이 수 {tokens_per_cell(train, False):.2f} -> (통일·중복 제거 후) {tokens_per_cell(train, True):.2f}")


train 전체 토큰 255,164개 중 '숫자+X' 종결 표기: 0개
Series([], Name: count, dtype: int64)
p. 접두 표기: 0 | 3글자 아미노산 표기: 0


train 칸당 변이 수 1.17 -> (통일·중복 제거 후) 1.14


In [9]:
# ---- 경로/유전자군 사전 (초안: 필요에 맞게 수정/추가) ----
PATHWAYS = {
    "p53": ["TP53","MDM2","MDM4","CDKN2A","ATM","CHEK1","CHEK2"],
    "RTK_RAS": ["EGFR","ERBB2","ERBB3","ERBB4","FGFR1","FGFR2","FGFR3","FGFR4","MET","ALK","RET","KIT",
                "PDGFRA","KRAS","NRAS","HRAS","BRAF","RAF1","NF1","MAP2K1","MAP2K2","SOS1","PTPN11"],
    "PI3K": ["PIK3CA","PIK3CB","PIK3R1","PTEN","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2","STK11","RICTOR","INPP4B"],
    "Wnt": ["APC","CTNNB1","AXIN1","AXIN2","RNF43","ZNRF3","TCF7L2","GSK3B","AMER1"],
    "CellCycle": ["RB1","CDK4","CDK6","CCND1","CCND2","CCND3","CCNE1","CDKN1A","CDKN1B","CDKN2B","E2F1","FBXW7"],
    "Notch": ["NOTCH1","NOTCH2","NOTCH3","NOTCH4","FBXW7","SPEN"],
    "Hippo": ["NF2","LATS1","LATS2","SAV1","YAP1","WWTR1","TEAD1"],
    "TGFb": ["TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4","ACVR1B","ACVR2A"],
    "NRF2": ["NFE2L2","KEAP1","CUL3"],
    "SWI_SNF": ["ARID1A","ARID1B","ARID2","SMARCA4","SMARCB1","PBRM1","SMARCA2"],
    "Chromatin": ["KMT2A","KMT2B","KMT2C","KMT2D","EZH2","SETD2","KDM6A","KDM5C","CREBBP","EP300",
                  "BAP1","ASXL1","DNMT3A","IDH1","IDH2","ATRX"],
    "DDR_MMR": ["BRCA1","BRCA2","PALB2","ATR","POLE","POLD1","MLH1","MSH2","MSH6","PMS2",
                "RAD51","RAD50","MRE11","FANCA","FANCD2","ERCC2"],
    "Splicing": ["SF3B1","SRSF2","U2AF1","RBM10"],
    "LongGenes": ["TTN","MUC16","MUC4","SYNE1","OBSCN","RYR2","RYR1","CSMD1","CSMD3","LRP1B","USH2A","FLG","PCLO"],
}
gene_pos = {g: i for i, g in enumerate(gene_cols)}
PW_IDX = {k: [gene_pos[g] for g in v if g in gene_pos] for k, v in PATHWAYS.items()}
print({k: f"{len(v)}/{len(PATHWAYS[k])}" for k, v in PW_IDX.items()})   # 데이터에 존재하는 유전자 수/사전 크기

{'p53': '5/7', 'RTK_RAS': '14/23', 'PI3K': '11/13', 'Wnt': '6/9', 'CellCycle': '12/12', 'Notch': '5/6', 'Hippo': '4/7', 'TGFb': '6/7', 'NRF2': '2/3', 'SWI_SNF': '0/7', 'Chromatin': '6/16', 'DDR_MMR': '11/16', 'Splicing': '2/4', 'LongGenes': '4/13'}


In [10]:
# ---- 유전자/토큰/아미노산쌍 사전은 "학습에 쓰는 행"에서만 만든다 (Data Leakage / CV 왜곡 방지) ----
# CV에서는 fold의 train 행만으로 사전을 만들고, 그 fold의 valid/test는 같은 사전으로 변환만 한다.
# test에만 있는 토큰은 tok_idx.get()이 None이라 무시된다.
def uniq_pairs(ex, in_rows):
    return set((r, nm) for r, nm in zip(ex["tok_rows"].tolist(), ex["tok_names"]) if in_rows[r])

def fit_vocab(ex, rows):
    rows = np.asarray(rows)
    in_rows = np.zeros(ex["mut"].shape[0], bool); in_rows[rows] = True
    keep_idx = np.where(ex["mut"][rows].sum(0) >= CFG["MIN_GENE_FREQ"])[0]
    cnt = Counter(nm for _, nm in uniq_pairs(ex, in_rows))
    keep_tokens = sorted(t for t, c in cnt.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
    aa_keep = np.where(ex["aa_cnt"][rows].sum(0) >= CFG["MIN_AA_FREQ"])[0]
    return dict(keep_idx=keep_idx, keep_tokens=keep_tokens,
                tok_idx={t: i for i, t in enumerate(keep_tokens)}, aa_keep=aa_keep)

def compose(ex, voc):
    mut = ex["mut"]; n = mut.shape[0]
    blocks, names = [], []
    def add(a, nm):
        blocks.append(np.asarray(a, np.float32).reshape(n, -1)); names.extend(nm)

    if CFG["GENE_MODE"] in ("all", "both"):
        add(mut[:, voc["keep_idx"]], [f"g_{gene_cols[i]}" for i in voc["keep_idx"]])
    if CFG["GENE_MODE"] in ("nonsilent", "both"):     # silent 변이만 있는 칸은 0
        add(ex["ns"][:, voc["keep_idx"]], [f"gns_{gene_cols[i]}" for i in voc["keep_idx"]])

    n_genes = mut.sum(1)
    tc = ex["type_cnt"]; n_ev = tc.sum(1)
    add(np.c_[n_genes, np.log1p(n_genes), n_ev], ["n_mut_genes", "log_n_mut_genes", "n_events"])
    if CFG["USE_SPARSE_META"]:   # 정보량이 극히 적은 샘플 표시 (모델이 과신하지 않도록)
        add(np.c_[n_genes == 0, n_genes <= 2, n_genes <= CFG["TMB_LOW"]], ["sp_tmb0", "sp_tmb_le2", "sp_tmb_le_low"])
    add(tc, [f"cnt_{t}" for t in TYPES])
    trunc = tc[:, [TYPE_IDX[t] for t in ("nonsense", "frameshift", "splice")]].sum(1)
    add(np.c_[tc[:, TYPE_IDX["silent"]] / (n_ev + 1),
              tc[:, TYPE_IDX["missense"]] / (n_ev + 1),
              trunc / (n_ev + 1)], ["ratio_silent", "ratio_missense", "ratio_trunc"])

    if CFG["USE_HOTSPOTS"] and voc["keep_tokens"]:
        H = np.zeros((n, len(voc["keep_tokens"])), np.float32)
        for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
            j = voc["tok_idx"].get(nm)
            if j is not None:
                H[r, j] = 1
        add(H, [f"hs_{t}" for t in voc["keep_tokens"]])
    if CFG["USE_COMBOS"]:
        C, cn = combo_features(ex)
        add(C, [f"cb_{k}" for k in cn])
    if CFG["USE_AA_SUBS"] and len(voc["aa_keep"]):
        A = ex["aa_cnt"][:, voc["aa_keep"]] / (tc[:, TYPE_IDX["missense"]] + 1)[:, None]
        add(A, [f"aa_{AA[k // 20]}>{AA[k % 20]}" for k in voc["aa_keep"]])

    if CFG["USE_PATHWAYS"]:
        P = np.c_[[mut[:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
        add(P, [f"pw_{k}" for k in PW_IDX])

    return np.hstack(blocks), names

# 참고용(진단): 전체 train 기준 사전 크기. 모델 학습에는 쓰지 않고 fold마다 fit_vocab을 다시 호출한다.
_v = fit_vocab(ex_tr, np.arange(len(train)))
print(f"(참고) 전체 train 기준 유전자 {len(_v['keep_idx'])}/{len(gene_cols)}, 핫스팟 토큰 {len(_v['keep_tokens'])}, 아미노산쌍 {len(_v['aa_keep'])}")


(참고) 전체 train 기준 유전자 4171/4384, 핫스팟 토큰 1795, 아미노산쌍 158


### 4-0. 프로필 중복·라벨 충돌 진단 + 그룹 분할 준비

같은 환자가 개별 코호트(`KIRC`)와 통합 코호트(`KIPAN`)에 모두 등록되면 변이 프로필이 완전히 같은 행이 생긴다 (`LGG`↔`GBMLGG`도 동일). 이를 **후처리로 덮어쓰지 않고**, 먼저 규모를 진단하고 분할 단계에서 같은 프로필이 학습/검증에 갈라지지 않게 묶는다.
- 그룹 = 변이 프로필 해시. `StratifiedGroupKFold`로 클래스 비율은 유지하면서 같은 그룹은 같은 fold로 보낸다.
- 변이가 거의 없는 프로필(예: 전부 WT)은 우연히 같을 수 있으므로, 진단에서 변이 유전자 수 기준(`TMB_LOW`)으로 따로 센다.


In [11]:
import hashlib

def profile_hashes(df):
    """행마다 변이 프로필(유전자 셀 전부)의 지문. 셀이 전부 같으면 지문이 같다."""
    return np.array([hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                     for r in df[gene_cols].to_numpy(dtype=object)])

H_tr, H_te = profile_hashes(train), profile_hashes(test)
lab_tr = train["SUBCLASS"].values

# ---- 진단: 같은 프로필 그룹과 라벨 충돌 ----
SIBLINGS = [frozenset(("KIRC", "KIPAN")), frozenset(("LGG", "GBMLGG"))]
g = pd.DataFrame({"h": H_tr, "y": lab_tr, "tmb": (train[gene_cols] != "WT").sum(1).values})
grp = g.groupby("h").agg(n=("y", "size"), labels=("y", lambda s: frozenset(s)), tmb=("tmb", "first"))
dup = grp[grp["n"] > 1]
conf = dup[dup["labels"].map(len) > 1]
sib = conf[conf["labels"].isin(SIBLINGS)]
print(f"train 프로필 그룹 {len(grp):,}개 중 2행 이상 중복: {len(dup):,}개 (해당 행 {int(dup['n'].sum()):,}개, 전체의 {dup['n'].sum()/len(train):.1%})")
print(f"  - 변이 유전자 {CFG['TMB_LOW']}개 이상인 중복 그룹(우연일 가능성 낮음): {int((dup['tmb'] >= CFG['TMB_LOW']).sum())}개")
print(f"  - 서로 다른 라벨이 섞인 그룹: {len(conf)}개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): {len(sib)}개, 그 외: {len(conf) - len(sib)}개")
if len(conf) - len(sib):
    print("  - 형제 외 라벨 충돌 예:", [tuple(sorted(s)) for s in conf[~conf['labels'].isin(SIBLINGS)]['labels'].head(5)])
in_tr = pd.Series(H_te).isin(set(H_tr))
print(f"test 행 중 train과 프로필이 같은 행: {int(in_tr.sum())}개 ({in_tr.mean():.1%})  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)")

# ---- 분할: 같은 프로필은 같은 fold로 ----
GRP_SIZE = pd.Series(H_tr).map(pd.Series(H_tr).value_counts()).to_numpy(dtype=float)   # 행마다 같은 프로필 행 수

def make_folds(group):
    if group:
        sp = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
        return list(sp.split(np.zeros(len(y)), y, groups=H_tr))
    sp = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    return list(sp.split(np.zeros(len(y)), y))

FOLDS_RANDOM, FOLDS_GROUP = make_folds(False), make_folds(True)
assert all(not (set(H_tr[tr]) & set(H_tr[va])) for tr, va in FOLDS_GROUP), "그룹 분할 실패: 같은 프로필이 train/valid에 걸쳐 있음"
FOLDS = FOLDS_GROUP if CFG["GROUP_CV"] else FOLDS_RANDOM
print("주 CV 분할:", "그룹(프로필) 분할" if CFG["GROUP_CV"] else "무작위 분할", "| fold 크기:", [len(va) for _, va in FOLDS])


train 프로필 그룹 5,636개 중 2행 이상 중복: 451개 (해당 행 1,016개, 전체의 16.4%)
  - 변이 유전자 5개 이상인 중복 그룹(우연일 가능성 낮음): 394개
  - 서로 다른 라벨이 섞인 그룹: 447개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): 445개, 그 외: 2개
  - 형제 외 라벨 충돌 예: [('LGG', 'THCA'), ('BRCA', 'CESC', 'GBMLGG', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LUSC', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'STES', 'THCA', 'THYM')]
test 행 중 train과 프로필이 같은 행: 266개 (10.4%)  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)


주 CV 분할: 무작위 분할 | fold 크기: [1241, 1240, 1240, 1240, 1240]


### 4-1. KNN 유사도 피처 (희소 변이 샘플 보강)

변이가 적은 샘플은 트리 모델이 분기 조건을 잡기 어렵다. 그래서 변이 유전자 집합의 **Jaccard 유사도**로 가장 가까운 train 환자 `KNN_K`명을 찾아 그들의 암종 분포를 피처로 준다.
- 이웃은 fold의 train 행에서만 찾는다. train 행 자신의 피처는 자기를 뺀 나머지에서, valid/test 행은 같은 train 행들에서 계산한다 (stacking의 leave-one-out 방식).
- **같은 프로필(해시 동일) 행은 이웃에서 뺀다.** 안 빼면 쌍둥이의 짝 라벨이 최근접 이웃으로 들어와 모델이 그 패턴을 몰래 학습한다(= 쌍둥이 규칙을 모델 안에 숨기는 셈). 빼면 "비슷하지만 같지는 않은" 환자의 분포만 쓴다.


In [12]:
from scipy import sparse

# (claude) train 해시만으로 번호표를 만들고 test는 조회만 한다. train에 없는 test 프로필은 -1 (어떤 train 행과도 다름)
ID_TR, _uniq = pd.factorize(H_tr)
_id_map = {h: i for i, h in enumerate(_uniq)}
ID_TE = np.array([_id_map.get(h, -1) for h in H_te])
KNN_NAMES = [f"knn_p_{c}" for c in le.classes_] + ["knn_max_sim", "knn_mean_sim"]

def knn_features(Q, q_id, R, r_id, r_y, k, chunk=512):
    """Q 행마다 R(참조 집합)에서 Jaccard 유사도 상위 k명의 암종 분포(유사도 가중) + 최대/평균 유사도.
    q_id == r_id 인 참조 행(자기 자신, 같은 프로필의 쌍둥이/중복)은 이웃에서 제외."""
    Qs = sparse.csr_matrix(np.asarray(Q) > 0, dtype=np.float32)
    Rs = sparse.csr_matrix(np.asarray(R) > 0, dtype=np.float32)
    nq = np.asarray(Qs.sum(1)).ravel(); nr = np.asarray(Rs.sum(1)).ravel()
    k = max(1, min(k, Rs.shape[0] - 1))
    out = np.zeros((Qs.shape[0], n_class + 2), np.float32)
    for s in range(0, Qs.shape[0], chunk):
        e = min(s + chunk, Qs.shape[0])
        inter = (Qs[s:e] @ Rs.T).toarray()
        union = nq[s:e, None] + nr[None, :] - inter
        sim = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        sim[q_id[s:e, None] == r_id[None, :]] = -1.0
        idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]
        ss = np.take_along_axis(sim, idx, 1).clip(min=0)
        lab = r_y[idx]
        for c in range(n_class):
            out[s:e, c] = (ss * (lab == c)).sum(1)
        tot = ss.sum(1, keepdims=True)
        out[s:e, :n_class] /= np.where(tot > 0, tot, 1.0)
        out[s:e, n_class] = ss.max(1); out[s:e, n_class + 1] = ss.mean(1)
    return out

# ---- 동작 확인 (작은 예제) ----
_n_class_bak = n_class; n_class = 3
_Rm = np.array([[1,1,0,0],[1,1,0,0],[1,0,1,0],[0,0,1,1],[0,0,0,0]], bool)   # 0번과 1번은 같은 프로필(쌍둥이)
_ry = np.array([0, 1, 0, 2, 2]); _rid = np.array([0, 0, 1, 2, 3])
_o = knn_features(_Rm, _rid, _Rm, _rid, _ry, k=2)
assert np.allclose(_o[0, :3], [1, 0, 0])      # 0번의 이웃: 같은 프로필인 1번(라벨1)은 제외 -> 2번(라벨0)이 가장 가까움
assert np.allclose(_o[1, :3], [1, 0, 0])      # 1번도 마찬가지 (쌍둥이 라벨이 새어 들어오지 않음)
assert np.allclose(_o[4, :3], 0)              # 변이 없는 행은 유사도 0 -> 분포 0 (가짜 확신 없음)
n_class = _n_class_bak
print("KNN 피처 동작 확인 통과, 피처 수:", len(KNN_NAMES))


KNN 피처 동작 확인 통과, 피처 수: 28


## 4. 5-fold CV (XGBoost + Logistic Regression)

※ XGB는 검증 fold로 early stopping을 하므로 OOF 점수가 아주 조금 낙관적일 수 있습니다 (모델 간 비교용으로는 충분).
※ 유전자/핫스팟/아미노산쌍 선택(빈도 필터)은 fold의 train 행만으로 매 fold 다시 하고, valid/test에는 그 사전으로 변환만 한다.
※ 분할은 4-0절의 `FOLDS`(GROUP_CV면 같은 프로필끼리 묶음)를 쓴다. 이전 무작위 분할보다 점수가 낮아져도 그만큼 중복에 의한 낙관이 빠진 것이다.


In [13]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# ---------------- 잡음 생성 (fold의 train 행 통계만 사용) ----------------
def fold_noise_params(ex, tr):
    """가짜 변이를 뽑을 분포: 유전자 빈도, 치환쌍 빈도, silent 비율. 모두 fold의 train 행에서만 계산."""
    gp = ex["mut"][tr].sum(0).astype(float) + 1.0; gp /= gp.sum()
    ap = ex["aa_cnt"][tr].sum(0).astype(float) + 1e-3; ap /= ap.sum()
    tc = ex["type_cnt"][tr].sum(0)
    p_sil = tc[TYPE_IDX["silent"]] / max(tc[TYPE_IDX["silent"]] + tc[TYPE_IDX["missense"]], 1)
    return gp, ap, float(p_sil)

def noisy_sub_ex(ex, rows, lam, rng, params):
    """rows 행만 뽑아 행마다 Poisson(lam)개 유전자에 가짜 silent/missense 변이를 더한 ex. (행 독립 연산)"""
    gp, ap, p_sil = params
    rows = np.asarray(rows); n = len(rows); G = ex["mut"].shape[1]
    mut = ex["mut"][rows].copy(); ns = ex["ns"][rows].copy()
    tc = ex["type_cnt"][rows].copy(); aa = ex["aa_cnt"][rows].copy()
    for i, k in enumerate(rng.poisson(lam, n)):
        if k == 0:
            continue
        cand = rng.choice(G, size=min(G, 2 * k + 5), replace=False, p=gp)
        cand = cand[~mut[i, cand]][:k]
        sil = rng.random(len(cand)) < p_sil
        mut[i, cand] = True; ns[i, cand[~sil]] = True
        tc[i, TYPE_IDX["silent"]] += sil.sum(); tc[i, TYPE_IDX["missense"]] += (~sil).sum()
        if (~sil).sum():
            np.add.at(aa[i], rng.choice(400, size=int((~sil).sum()), p=ap), 1)
    pos = np.full(ex["mut"].shape[0], -1); pos[rows] = np.arange(n)
    sel = pos[ex["tok_rows"]] >= 0
    return dict(mut=mut, ns=ns, type_cnt=tc, aa_cnt=aa, tok_rows=pos[ex["tok_rows"][sel]],
                tok_names=[nm for nm, s in zip(ex["tok_names"], sel) if s])

# ---------------- 모델 ----------------
def fit_tree(kind, Xtr, ytr, wtr, Xes, yes, seed, n_fixed=None):
    """n_fixed=None이면 (Xes, yes)로 early stopping, 아니면 반복 수 고정. (모델, 사용 반복 수) 반환."""
    es = n_fixed is None
    if kind == "xgb":
        m = xgb.XGBClassifier(objective="multi:softprob", n_estimators=CFG["XGB_N_EST"] if es else n_fixed,
                              learning_rate=CFG["XGB_LR"], max_depth=5, subsample=0.8, colsample_bytree=0.3,
                              tree_method="hist", eval_metric="mlogloss", early_stopping_rounds=50 if es else None,
                              random_state=seed, n_jobs=-1)
        if es:
            m.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xes, yes)], verbose=False); return m, m.best_iteration + 1
        m.fit(Xtr, ytr, sample_weight=wtr, verbose=False); return m, n_fixed
    m = lgb.LGBMClassifier(objective="multiclass", n_estimators=CFG["XGB_N_EST"] if es else n_fixed,
                           learning_rate=CFG["LGB_LR"], num_leaves=31, colsample_bytree=0.3, subsample=0.8,
                           subsample_freq=1, min_child_samples=10, reg_lambda=1.0, random_state=seed,
                           n_jobs=-1, verbose=-1)
    if es:
        m.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)])
        return m, m.best_iteration_
    m.fit(Xtr, ytr, sample_weight=wtr); return m, n_fixed

def make_lr(C):
    return make_pipeline(MaxAbsScaler(), LogisticRegression(        # 스케일러는 학습 행으로만 fit
        C=C, max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))

def train_set(ex, rows, X, seed, voc, names):
    """학습 행렬/라벨/트리 가중치/LR 가중치. AUG_NOISE면 학습 행의 잡음 복사본을 붙인다 (학습 행 통계만 사용)."""
    sw = compute_sample_weight("balanced", y[rows]) if CFG["BALANCE"] else None
    pw = None
    if CFG["PROFILE_WEIGHT"]:
        pw = 1.0 / GRP_SIZE[rows]; pw = pw / pw.mean()
    if CFG["SPARSE_WEIGHT"] != 1.0:
        lw = np.where(ex["mut"][rows].sum(1) <= 2, CFG["SPARSE_WEIGHT"], 1.0)
        pw = lw if pw is None else pw * lw
    if pw is not None:
        sw = pw if sw is None else sw * pw
    Xtr, ytr = X[rows], y[rows]
    if CFG["AUG_NOISE"]:
        Xa, na = compose(noisy_sub_ex(ex, rows, CFG["AUG_MEAN"], np.random.default_rng(seed),
                                      fold_noise_params(ex, rows)), voc)
        assert na == names
        Xtr = np.vstack([Xtr, Xa]); ytr = np.r_[ytr, ytr]
        base_w = sw if sw is not None else np.ones(len(rows))
        sw = np.r_[base_w, base_w * CFG["AUG_WEIGHT"]]
        base_l = pw if pw is not None else np.ones(len(rows))
        pw = np.r_[base_l, base_l * CFG["AUG_WEIGHT"]]
    return Xtr, ytr, sw, pw

def run_cv(ex_tr, ex_te, verbose=True, folds=None):
    assert not (CFG["USE_KNN"] and (CFG["AUG_NOISE"] or CFG["NOISE_EVAL"] or CFG["FULL_REFIT"])), "KNN과 잡음/전체 재학습 동시 사용은 미구현"
    folds = FOLDS if folds is None else folds
    n_te = ex_te["mut"].shape[0]
    kinds = ["xgb"] + (["lgb"] if CFG["USE_LGB"] else [])
    Z = lambda n: np.zeros((n, n_class))
    oof = {k: Z(len(y)) for k in kinds}; te = {k: Z(n_te) for k in kinds}; noi = {k: Z(len(y)) for k in kinds}
    oof_lr = {C: Z(len(y)) for C in CFG["LR_CS"]}; te_lr = {C: Z(n_te) for C in CFG["LR_CS"]}; noi_lr = {C: Z(len(y)) for C in CFG["LR_CS"]}
    iters = {k: [] for k in kinds}
    imp = Counter()

    for fold, (tr, va) in enumerate(folds):
        voc = fit_vocab(ex_tr, tr)                      # 사전은 이 fold의 train 행만으로
        X_all, names = compose(ex_tr, voc)
        X_te, _ = compose(ex_te, voc)                   # test는 같은 사전으로 변환만
        if CFG["USE_KNN"]:
            cols = voc["keep_idx"]
            R, r_id, r_y = ex_tr["mut"][tr][:, cols], ID_TR[tr], y[tr]
            X_all = np.hstack([X_all, knn_features(ex_tr["mut"][:, cols], ID_TR, R, r_id, r_y, CFG["KNN_K"])])
            X_te = np.hstack([X_te, knn_features(ex_te["mut"][:, cols], ID_TE, R, r_id, r_y, CFG["KNN_K"])])
            names = names + KNN_NAMES
        X = X_all
        assert len(np.unique(y[tr])) == n_class, "일부 클래스가 train fold에 없습니다 (N_FOLDS를 줄이세요)"
        Xtr, ytr, wtr, lrw = train_set(ex_tr, tr, X, CFG["SEED"] + 2000 + fold, voc, names)
        Xn = None
        if CFG["NOISE_EVAL"]:                           # 평가용 잡음: 검증 행에만, 다른 seed
            Xn, nn_ = compose(noisy_sub_ex(ex_tr, va, CFG["NOISE_MEAN"], np.random.default_rng(CFG["SEED"] + 1000 + fold),
                                           fold_noise_params(ex_tr, tr)), voc)
            assert nn_ == names

        for kind in kinds:
            for seed in CFG["XGB_SEEDS"]:
                if CFG["INNER_ES"]:                     # 검증 fold는 보지 않고, 학습 fold 안에서 반복 수 결정
                    i_tr, i_es = train_test_split(np.arange(len(tr)), test_size=0.15, stratify=y[tr], random_state=seed + fold)
                    i_fit = np.r_[i_tr, i_tr + len(tr)] if CFG["AUG_NOISE"] else i_tr
                    _, bi = fit_tree(kind, Xtr[i_fit], ytr[i_fit], None if wtr is None else wtr[i_fit],
                                     Xtr[i_es], ytr[i_es], seed)
                    m, n_use = fit_tree(kind, Xtr, ytr, wtr, None, None, seed, n_fixed=max(10, int(round(bi * CFG["REFIT_MULT"]))))
                else:                                   # 민정 v10 방식: 검증 fold로 early stopping
                    m, n_use = fit_tree(kind, Xtr, ytr, wtr, X[va], y[va], seed)
                iters[kind].append(n_use)
                ns_ = len(CFG["XGB_SEEDS"])
                oof[kind][va] += m.predict_proba(X[va]) / ns_
                te[kind] += m.predict_proba(X_te) / ns_ / CFG["N_FOLDS"]
                if Xn is not None:
                    noi[kind][va] += m.predict_proba(Xn) / ns_
                if kind == "xgb":
                    for nm_, v_ in zip(names, m.feature_importances_): imp[nm_] += v_ / CFG["N_FOLDS"] / ns_

        for C in CFG["LR_CS"]:
            lr = make_lr(C)
            if lrw is None: lr.fit(Xtr, ytr)
            else: lr.fit(Xtr, ytr, logisticregression__sample_weight=lrw)
            oof_lr[C][va] = lr.predict_proba(X[va]); te_lr[C] += lr.predict_proba(X_te) / CFG["N_FOLDS"]
            if Xn is not None: noi_lr[C][va] = lr.predict_proba(Xn)

        if verbose:
            f = lambda p: f1_score(y[va], p[va].argmax(1), average="macro")
            msg = " | ".join(f"{k.upper()}(iter={iters[k][-1]}) {f(oof[k]):.4f}" for k in kinds)
            print(f"fold{fold} | {msg} | " + " ".join(f"LR(C={C}) {f(oof_lr[C]):.4f}" for C in CFG["LR_CS"]))

    # LR C 선택 (후보가 여러 개일 때만, OOF 기준)
    bestC = max(CFG["LR_CS"], key=lambda C: f1_score(y, oof_lr[C].argmax(1), average="macro"))
    oof["lr"], te["lr"], noi["lr"] = oof_lr[bestC], te_lr[bestC], noi_lr[bestC]
    if len(CFG["LR_CS"]) > 1:
        print("LR C 후보 OOF:", {C: round(f1_score(y, oof_lr[C].argmax(1), average="macro"), 4) for C in CFG["LR_CS"]}, "-> 선택", bestC)

    if CFG["FULL_REFIT"]:                               # test 예측: train 전체로 재학습 (반복 수는 fold 내부 분할 결과로)
        allr = np.arange(len(y))
        voc = fit_vocab(ex_tr, allr); X_all, names = compose(ex_tr, voc); X_te, _ = compose(ex_te, voc)
        Xtr, ytr, wtr, lrw = train_set(ex_tr, allr, X_all, CFG["SEED"] + 3000, voc, names)
        for kind in kinds:
            te[kind] = Z(n_te); n_full = int(round(np.mean(iters[kind]) * CFG["FULL_MULT"]))
            for seed in CFG["XGB_SEEDS"]:
                m, _ = fit_tree(kind, Xtr, ytr, wtr, None, None, seed, n_fixed=n_full)
                te[kind] += m.predict_proba(X_te) / len(CFG["XGB_SEEDS"])
            print(f"전체 재학습 {kind}: n_estimators={n_full}")
        lr = make_lr(bestC)
        if lrw is None: lr.fit(Xtr, ytr)
        else: lr.fit(Xtr, ytr, logisticregression__sample_weight=lrw)
        te["lr"] = lr.predict_proba(X_te)
    return dict(oof=oof, te=te, noise=noi if CFG["NOISE_EVAL"] else None, imp=imp, iters=iters, lr_C=bestC)

_T_CV = time.time()
RES = run_cv(ex_tr, ex_te)
print(f"CV 시간 {time.time() - _T_CV:.0f}s")


fold0 | XGB(iter=240) 0.4336 | LR(C=0.1) 0.4225


fold1 | XGB(iter=201) 0.4632 | LR(C=0.1) 0.4135


fold2 | XGB(iter=256) 0.4520 | LR(C=0.1) 0.4218


fold3 | XGB(iter=212) 0.4695 | LR(C=0.1) 0.4182


fold4 | XGB(iter=230) 0.4563 | LR(C=0.1) 0.4364


전체 재학습 xgb: n_estimators=251


CV 시간 604s


## 5. 평가 · 앙상블 · 오분류 분석

In [14]:
import itertools
def report(name, p):
    print(f"{name:12s} acc={accuracy_score(y, p.argmax(1)):.4f}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}")

MODELS = list(RES["oof"])          # ["xgb", (lgb), "lr"]
for k in MODELS: report(k.upper(), RES["oof"][k])

def blend(w, src):
    return sum(w[k] * src[k] for k in MODELS)

# 가중치 격자 (0.1 단위, 합 1). 모델 2개면 민정 v10과 같은 11점 격자.
best_w, best_s = None, -1
for ws in itertools.product(np.round(np.linspace(0, 1, 11), 1), repeat=len(MODELS) - 1):
    if sum(ws) > 1 + 1e-9: continue
    w = dict(zip(MODELS, list(ws) + [round(1 - sum(ws), 1)]))
    s = f1_score(y, blend(w, RES["oof"]).argmax(1), average="macro")
    if s > best_s: best_w, best_s = w, s
oof_ens, te_ens = blend(best_w, RES["oof"]), blend(best_w, RES["te"])
noise_ens = blend(best_w, RES["noise"]) if RES["noise"] is not None else None
print(f"\n앙상블 가중치={best_w}")
report("ENSEMBLE", oof_ens)
if noise_ens is not None: report("ENS(noise)", noise_ens)

print("\n" + classification_report(y, oof_ens.argmax(1), target_names=le.classes_, digits=3))
cm = confusion_matrix(y, oof_ens.argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], le.classes_[i], le.classes_[j]) for i in range(n_class) for j in range(n_class) if cm[i, j] > 0), reverse=True)[:10]
print("가장 많이 헷갈리는 쌍 (정답 -> 예측):")
for c, a, b in pairs: print(f"  {a} -> {b}: {c}")
print("\nXGB 중요도 상위 20:", [k for k, _ in RES["imp"].most_common(20)])


XGB          acc=0.4391  macroF1=0.4556
LR           acc=0.4127  macroF1=0.4239

앙상블 가중치={'xgb': np.float64(0.3), 'lr': np.float64(0.7)}
ENSEMBLE     acc=0.4515  macroF1=0.4709
ENS(noise)   acc=0.3166  macroF1=0.3025

              precision    recall  f1-score   support

         ACC      0.932     0.764     0.840        72
        BLCA      0.661     0.394     0.494       104
        BRCA      0.753     0.426     0.544       786
        CESC      0.357     0.426     0.388       155
        COAD      0.782     0.722     0.751       223
        DLBC      0.786     0.289     0.423        38
      GBMLGG      0.435     0.262     0.327       461
        HNSC      0.417     0.386     0.401       223
       KIPAN      0.238     0.219     0.229       515
        KIRC      0.149     0.159     0.154       334
        LAML      0.592     0.487     0.535       158
         LGG      0.358     0.546     0.433       229
        LIHC      0.504     0.437     0.468       158
        LUAD      0.496  

In [15]:
# COAD 비율 보정: 검토 결과 미적용(k=1.0) → test 정답 분포 정보를 쓰지 않기 위해 삭제 (11/12)
te_final = te_ens.copy()


In [16]:
# ---- 현업 기준 점수: 같은 프로필을 같은 fold에 묶은 그룹 분할 OOF ----
honest = None
if CFG["HONEST_REPORT"] and not CFG["GROUP_CV"]:
    _keep_full = CFG["FULL_REFIT"]; CFG["FULL_REFIT"] = False
    _G = run_cv(ex_tr, ex_te, verbose=False, folds=FOLDS_GROUP)
    CFG["FULL_REFIT"] = _keep_full
    _ge = blend(best_w, _G["oof"])
    honest = dict(macroF1=float(f1_score(y, _ge.argmax(1), average="macro")), acc=float(accuracy_score(y, _ge.argmax(1))))
    print(f"[현업 기준] 그룹 분할 OOF  macroF1={honest['macroF1']:.4f}  acc={honest['acc']:.4f}")
else:
    print("그룹 분할 점수 생략 (HONEST_REPORT=False)")


그룹 분할 점수 생략 (HONEST_REPORT=False)


## 5-2. 학습형 중복 규칙 (`USE_DUP_RULE`)

- 같은 환자가 개별 코호트(`KIRC`)와 통합 코호트(`KIPAN`)에 모두 등록되면 변이 프로필이 같은 행이 둘 생긴다. 한 행만 train에 있으면 모델은 그 라벨을 그대로 찍어 틀린다 (fixed4 출력: 해당 행 정답률 0.096).
- 이전 규칙은 암종 이름을 코드에 박아 둔 `TWIN_SWAP`이었다. 여기서는 **표를 train에서 추정**한다: "같은 프로필을 가진 *다른* train 행들의 라벨 묶음 -> 이 행의 라벨 분포"를 leave-one-out으로 센다. 한 칸의 표본 수가 `DUP_MIN_SUPPORT` 이상이고 최다 라벨 비율이 `DUP_MIN_PROB` 이상일 때만 쓴다.
- 장점: 암종 이름이 코드에 없다(어떤 쌍에도 적용), 근거가 부족하면 자동으로 꺼진다, 표를 CSV로 감사할 수 있다.
- **한계(솔직히)**: 효과의 출처는 이전과 같은 "이중 등록 구조"다. 새 환자에게는 이런 짝이 없으므로 **현업 성능이 아니다.** 그래서 제출 파일을 규칙 없는 `_core`와 규칙 적용 `_dup`으로 나눠 만든다. `_dup`을 내도 되는지는 대회 규정 확인이 필요하다.
- 검증: 주 CV가 무작위 분할일 때만 시뮬이 가능하다 (그룹 분할이면 짝이 검증에 남지 않아 발동 0). 사전은 fold의 train 행으로만 만들고 검증 행은 한 행씩 독립으로 찾아본다. test끼리는 비교하지 않는다.


In [17]:
def build_dup_table(h, lab):
    """같은 프로필 행들 중 '나를 뺀 나머지의 라벨 묶음' -> 내 라벨 분포 (leave-one-out)."""
    cnt = {}
    for labs in pd.Series(np.asarray(lab)).groupby(np.asarray(h)).agg(list):
        if len(labs) < 2 or len(labs) - 1 > CFG["DUP_MAX_CTX"]:
            continue
        for k, yk in enumerate(labs):
            cnt.setdefault(tuple(sorted(labs[:k] + labs[k + 1:])), Counter())[yk] += 1
    tab = {}
    for ctx, c in cnt.items():
        n = sum(c.values()); lab_, k_ = c.most_common(1)[0]
        if n >= CFG["DUP_MIN_SUPPORT"] and k_ / n >= CFG["DUP_MIN_PROB"]:
            tab[ctx] = (lab_, k_ / n, n)
    return tab

def dup_override(tab, tr_hash, tr_labels, q_hash):
    """질의 행마다 바꿀 암종 번호(없으면 -1). train 프로필->라벨 목록 사전을 만들고 한 행씩 찾아본다."""
    labs = pd.Series(np.asarray(tr_labels)).groupby(np.asarray(tr_hash)).agg(list).to_dict()
    cls = list(le.classes_); out = np.full(len(q_hash), -1)
    for i, h in enumerate(q_hash):
        L = labs.get(h)
        if L and len(L) <= CFG["DUP_MAX_CTX"]:
            d = tab.get(tuple(sorted(L)))
            if d is not None and d[0] in cls:
                out[i] = cls.index(d[0])
    return out

def apply_override(P, ov):
    """ov >= 0 인 행은 그 암종 확률을 1로(나머지 0), 나머지 행은 그대로."""
    Q = P.copy(); m = ov >= 0
    Q[m] = 0.0; Q[m, ov[m]] = 1.0
    return Q

# ---- 동작 확인 (작은 예제: 30쌍 KIRC/KIPAN, 25쌍 LGG/GBMLGG, 잡음 군집) ----
_keep = {k: CFG[k] for k in ("DUP_MIN_SUPPORT", "DUP_MIN_PROB", "DUP_MAX_CTX")}
CFG.update(DUP_MIN_SUPPORT=20, DUP_MIN_PROB=0.9, DUP_MAX_CTX=3)
_h, _l = [], []
for k in range(30): _h += [f"k{k}"] * 2; _l += ["KIRC", "KIPAN"]
for k in range(25): _h += [f"g{k}"] * 2; _l += ["LGG", "GBMLGG"]
for k in range(5):  _h += [f"x{k}"] * 2; _l += ["BRCA", "OV"]          # 표본 부족 -> 쓰이지 않아야 함
_tab = build_dup_table(_h, _l)
assert _tab[("KIRC",)][0] == "KIPAN" and _tab[("KIPAN",)][0] == "KIRC" and _tab[("LGG",)][0] == "GBMLGG"
assert ("BRCA",) not in _tab and ("OV",) not in _tab                   # 표본 수 미달 칸은 제외
_le_bak, le = le, type("L", (), {"classes_": np.array(["BRCA", "GBMLGG", "KIPAN", "KIRC", "LGG", "OV"])})()
_trh, _trl = ["a", "b", "b", "c"], ["KIRC", "KIRC", "KIPAN", "BRCA"]    # a: 짝 하나만 train에 있음 / b: 짝이 이미 둘 다 train / c: 표에 없음
assert dup_override(_tab, _trh, _trl, ["a", "b", "c", "z"]).tolist() == [2, -1, -1, -1]
le = _le_bak; CFG.update(_keep)
assert np.allclose(apply_override(np.full((3, 5), 0.2), np.array([-1, 2, 0])).sum(1), 1)

# ---- 시뮬(진단): 주 CV와 같은 FOLDS, fold의 학습 부분만으로 표를 만들어 검증 부분에 적용 ----
sim = oof_ens.copy(); hit_oof = np.zeros(len(y), bool)
for tr_i, va_i in FOLDS:
    tab_f = build_dup_table(H_tr[tr_i], lab_tr[tr_i])
    ov = dup_override(tab_f, H_tr[tr_i], lab_tr[tr_i], H_tr[va_i])
    sim[va_i] = apply_override(oof_ens[va_i], ov); hit_oof[va_i[ov >= 0]] = True
n_hit = int(hit_oof.sum())
f0, f1 = (f1_score(y, p.argmax(1), average="macro") for p in (oof_ens, sim))
print(f"중복 규칙 시뮬: OOF macroF1 {f0:.4f} -> {f1:.4f} ({f1 - f0:+.4f}) | 발동 {n_hit}행 ({n_hit/len(y):.1%})")
if n_hit:
    print(f"  발동 행 정답률 {(sim[hit_oof].argmax(1) == y[hit_oof]).mean():.3f} (규칙 전 {(oof_ens[hit_oof].argmax(1) == y[hit_oof]).mean():.3f})")
elif CFG["GROUP_CV"]:
    print("  (그룹 분할이라 같은 프로필이 검증 fold에 짝으로 남지 않음 -> 시뮬 불가. 정상)")

# ---- 전체 train으로 만든 표(감사용 출력)와 test 쪽 규모 ----
dup_tab = build_dup_table(H_tr, lab_tr)
print("학습된 중복 표 (같은 프로필의 다른 train 라벨 묶음 -> 예측 라벨, 비율, 표본 수):")
for ctx, (lb, p, n) in sorted(dup_tab.items(), key=lambda kv: -kv[1][2]):
    print(f"  {ctx} -> {lb}  p={p:.3f}  n={n}")
ov_te = dup_override(dup_tab, H_tr, lab_tr, H_te)
print(f"test에서 발동 가능한 행: {(ov_te >= 0).sum()}행 ({(ov_te >= 0).mean():.1%}) ->",
      pd.Series(le.classes_[ov_te[ov_te >= 0]]).value_counts().to_dict())
dup_applied = bool(CFG["USE_DUP_RULE"] and n_hit >= CFG["DUP_MIN_SUPPORT"] and (f1 - f0) >= CFG["BOOST_MIN_GAIN"])
print("중복 규칙", "적용 (제출 파일 _dup)" if dup_applied else "미적용 (OFF이거나 시뮬 이득/표본이 기준 미만)")


중복 규칙 시뮬: OOF macroF1 0.4709 -> 0.5368 (+0.0659) | 발동 716행 (11.5%)
  발동 행 정답률 0.997 (규칙 전 0.095)
학습된 중복 표 (같은 프로필의 다른 train 라벨 묶음 -> 예측 라벨, 비율, 표본 수):
  ('KIPAN',) -> KIRC  p=0.993  n=272
  ('KIRC',) -> KIPAN  p=1.000  n=270
  ('GBMLGG',) -> LGG  p=1.000  n=173
  ('LGG',) -> GBMLGG  p=1.000  n=173
test에서 발동 가능한 행: 220행 (8.6%) -> {'KIRC': 62, 'KIPAN': 58, 'GBMLGG': 52, 'LGG': 48}
중복 규칙 적용 (제출 파일 _dup)


## 5-3. macroF1 클래스별 확률 배율 보정 (`USE_CLASS_SCALE`)

macroF1은 클래스별 F1의 평균이라, 판정 기준을 클래스마다 조금씩 옮기면(확률에 배율을 곱한 뒤 argmax) 점수가 오를 수 있다. 의료 분류에서도 쓰는 표준적인 결정 임계값 조정이다.
- 배율은 **OOF 확률과 train 라벨만**으로 맞춘다 (test 분포 정보 사용 안 함. 이전에 뺀 EM 사전확률 보정과 다름).
- 과적합 방지: fold마다 "나머지 fold의 OOF로 배율을 맞추고 남은 fold로 평가"하는 **중첩 검증**을 하고, 그 이득이 `BOOST_MIN_GAIN` 이상일 때만 적용한다. 보고하는 이득은 이 보수적인 추정치다.
- 중복 규칙이 정답을 바로잡는 행은 배율을 맞출 때 제외한다 (규칙이 이미 해결하는 오류에 배율이 맞춰지는 것을 막기 위해).


In [18]:
def macro_f1(yt, pred, K):
    tp = np.bincount(yt[pred == yt], minlength=K).astype(float)
    pc, tc = np.bincount(pred, minlength=K), np.bincount(yt, minlength=K)
    d = pc + tc
    return float(np.mean(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)[d > 0]))

_r = np.random.default_rng(0); _yt = _r.integers(0, 6, 500); _pr = _r.integers(0, 6, 500)   # sklearn과 같은 값인지 확인
assert abs(macro_f1(_yt, _pr, 6) - f1_score(_yt, _pr, average="macro")) < 1e-9

SCALE_GRID = np.exp(np.linspace(-1.0, 1.0, 21))

def tune_scale(P, yy, iters):
    """클래스 하나씩 배율을 바꿔 보며 macroF1이 오르면 채택 (좌표 탐색). 기준은 배율 1.0."""
    K = P.shape[1]; s = np.ones(K); best = macro_f1(yy, P.argmax(1), K)
    for _ in range(iters):
        for c in range(K):
            for g in SCALE_GRID:
                if g == s[c]: continue
                t = s.copy(); t[c] = g
                v = macro_f1(yy, (P * t).argmax(1), K)
                if v > best + 1e-9: best, s = v, t
    return s, best

# ---- 중첩 검증: 다른 fold로 배율을 맞추고, 남은 fold의 (중복 규칙이 안 건드리는) 행으로 평가 ----
mask = ~hit_oof if dup_applied else np.ones(len(y), bool)
pred_raw, pred_scaled, y_eval = [], [], []
for tr_i, va_i in FOLDS:
    fit_rows = tr_i[mask[tr_i]]; ev_rows = va_i[mask[va_i]]
    s_f, _ = tune_scale(oof_ens[fit_rows], y[fit_rows], CFG["SCALE_ITERS"])
    pred_raw.append(oof_ens[ev_rows].argmax(1)); pred_scaled.append((oof_ens[ev_rows] * s_f).argmax(1)); y_eval.append(y[ev_rows])
pred_raw, pred_scaled, y_eval = map(np.concatenate, (pred_raw, pred_scaled, y_eval))
g0, g1 = macro_f1(y_eval, pred_raw, n_class), macro_f1(y_eval, pred_scaled, n_class)
scale_applied = bool(CFG["USE_CLASS_SCALE"] and (g1 - g0) >= CFG["BOOST_MIN_GAIN"])
print(f"클래스 배율 중첩 검증: macroF1 {g0:.4f} -> {g1:.4f} ({g1 - g0:+.4f}) | 적용: {scale_applied}")

scale = np.ones(n_class)
if scale_applied:
    scale, _s_in = tune_scale(oof_ens[mask], y[mask], CFG["SCALE_ITERS"])      # 최종 배율은 전체 OOF로
    print("배율(1.0 아님):", {le.classes_[c]: round(float(v), 2) for c, v in enumerate(scale) if abs(v - 1) > 1e-6})
oof_core, te_core = oof_ens * scale, te_ens * scale
te_dup = apply_override(te_core, ov_te) if dup_applied else te_core
oof_dup = apply_override(oof_core, np.where(hit_oof, sim.argmax(1), -1)) if dup_applied else oof_core
print(f"OOF macroF1 (참고): 앙상블 {f1_score(y, oof_ens.argmax(1), average='macro'):.4f} | 배율 적용 {f1_score(y, oof_core.argmax(1), average='macro'):.4f} (배율은 같은 OOF로 맞춘 값이라 낙관적, 위 중첩 검증값이 보수적 추정)"
      + (f" | +중복 규칙 {f1_score(y, oof_dup.argmax(1), average='macro'):.4f}" if dup_applied else ""))

# ---- 제출 파이프라인 전체의 중첩 OOF 추정 (중복 규칙: fold 표 / 배율: 다른 fold로 맞춤) ----
FINAL_OOF = sim.copy()
if scale_applied:
    for tr_i, va_i in FOLDS:
        fit_rows = tr_i[mask[tr_i]]
        s_f, _ = tune_scale(oof_ens[fit_rows], y[fit_rows], CFG["SCALE_ITERS"])
        FINAL_OOF[va_i] = apply_override(oof_ens[va_i] * s_f, np.where(hit_oof[va_i], sim[va_i].argmax(1), -1))
print(f"제출 파이프라인 중첩 OOF macroF1 = {f1_score(y, FINAL_OOF.argmax(1), average='macro'):.4f}")


클래스 배율 중첩 검증: macroF1 0.4937 -> 0.4882 (-0.0055) | 적용: False
OOF macroF1 (참고): 앙상블 0.4709 | 배율 적용 0.4709 (배율은 같은 OOF로 맞춘 값이라 낙관적, 위 중첩 검증값이 보수적 추정) | +중복 규칙 0.5368
제출 파이프라인 중첩 OOF macroF1 = 0.5368


## 5-4. [v3] 형제 암종 쌍 전용 확률 배율 (`USE_PAIR_BIAS`)
v10의 26개 클래스 배율은 중첩 검증에서 손해(-0.002)였다. 자유도가 너무 커서다. 여기서는 **중복 표(train에서 학습)에 나온 형제 쌍의 클래스만** 배율을 둔다 (암종 이름을 코드에 쓰지 않음).
- 배율은 중복 규칙이 적용된 OOF(`FINAL_OOF`)에서 맞춘다. 규칙으로 확정된 행은 one-hot이라 배율에 영향을 받지 않는다.
- 중첩 검증: 다른 fold로 맞추고 남은 fold에 적용한 OOF로 이득을 재고, `BOOST_MIN_GAIN` 이상일 때만 적용한다. test 정보는 쓰지 않는다.


In [19]:
pair_applied, pair_scale = False, np.ones(n_class)
if CFG["USE_PAIR_BIAS"]:
    _cls = list(le.classes_)
    PAIR_CLS = sorted({_cls.index(c) for ctx, (lb, _, _) in dup_tab.items() if len(ctx) == 1 for c in (ctx[0], lb) if c in _cls})
    print("배율 대상 클래스(중복 표에서 학습):", [_cls[c] for c in PAIR_CLS])

    def tune_pair(P, yy, iters):
        s = np.ones(n_class); best = macro_f1(yy, P.argmax(1), n_class)
        for _ in range(iters):
            for c in PAIR_CLS:
                for g in SCALE_GRID:
                    if g == s[c]: continue
                    t = s.copy(); t[c] = g
                    v = macro_f1(yy, (P * t).argmax(1), n_class)
                    if v > best + 1e-9: best, s = v, t
        return s, best

    nested = FINAL_OOF.copy()
    for tr_i, va_i in FOLDS:
        s_f, _ = tune_pair(FINAL_OOF[tr_i], y[tr_i], CFG["SCALE_ITERS"])
        nested[va_i] = FINAL_OOF[va_i] * s_f
    p0 = f1_score(y, FINAL_OOF.argmax(1), average="macro"); p1 = f1_score(y, nested.argmax(1), average="macro")
    pair_applied = bool((p1 - p0) >= CFG["BOOST_MIN_GAIN"])
    print(f"쌍 배율 중첩 검증: {p0:.4f} -> {p1:.4f} ({p1 - p0:+.4f}) | 적용: {pair_applied}")
    if pair_applied:
        pair_scale, _ = tune_pair(FINAL_OOF, y, CFG["SCALE_ITERS"])     # 최종 배율은 전체 OOF로
        print("쌍 배율:", {_cls[c]: round(float(pair_scale[c]), 2) for c in PAIR_CLS})
        FINAL_OOF = nested
        te_core = te_core * pair_scale
        te_dup = apply_override(te_core, ov_te) if dup_applied else te_core


## 6. 제출 파일 생성

In [20]:
import json, sys, platform, datetime

tag = "norm" if CFG["NORMALIZE_TOKENS"] else "nonorm"

def save_submission(P, suffix):
    pred = le.inverse_transform(P.argmax(1))
    out = sub.copy(); out["SUBCLASS"] = pred
    assert len(out) == len(test) and set(out["SUBCLASS"]) <= set(le.classes_)
    path = f"{CFG['OUT_DIR']}/submission_{CFG['VERSION']}_{tag}_{suffix}.csv"
    out.to_csv(path, index=False, encoding="UTF-8-sig")
    np.save(f"{CFG['OUT_DIR']}/te_final_{CFG['VERSION']}_{tag}_{suffix}.npy", P)
    print(f"저장: {path}")
    return out

out_core = save_submission(te_core, "core")                 # 현업형: 모델 + (검증된) 클래스 배율. 쌍둥이/중복 덮어쓰기 없음
out_dup = save_submission(te_dup, "dup") if dup_applied else None   # 대회 조건형: + 학습형 중복 규칙
np.save(f"{CFG['OUT_DIR']}/oof_ens_{CFG['VERSION']}_{tag}.npy", oof_ens)
pd.DataFrame([(str(c), v[0], v[1], v[2]) for c, v in dup_tab.items()], columns=["context", "label", "prob", "n"]).to_csv(
    f"{CFG['OUT_DIR']}/dup_table_{CFG['VERSION']}_{tag}.csv", index=False, encoding="UTF-8-sig")

def _md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

manifest = dict(
    created=datetime.datetime.now().isoformat(timespec="seconds"),
    cfg={k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in CFG.items()},
    versions=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                  xgboost=xgb.__version__, platform=platform.platform(),
                  sklearn=__import__("sklearn").__version__),
    data=dict(train_md5=_md5(f"{D}/train.csv"), test_md5=_md5(f"{D}/test.csv"), n_train=len(train), n_test=len(test)),
    scores=dict(oof_ens_macroF1_main_cv=float(f1_score(y, oof_ens.argmax(1), average="macro")),
                honest_group_cv=honest, class_scale_nested=dict(before=g0, after=g1, applied=scale_applied),
                dup_rule_sim=dict(before=float(f0), after=float(f1), n_hit=n_hit, applied=dup_applied)),
    ensemble_weights=best_w, final_oof_nested=float(f1_score(y, FINAL_OOF.argmax(1), average="macro")),
    outputs=dict(core=out_core["SUBCLASS"].value_counts().to_dict(),
                 dup=(out_dup["SUBCLASS"].value_counts().to_dict() if out_dup is not None else None)),
)
with open(f"{CFG['OUT_DIR']}/run_manifest_{CFG['VERSION']}_{tag}.json", "w", encoding="utf-8") as fp:
    json.dump(manifest, fp, ensure_ascii=False, indent=2, default=str)
print("저장: run_manifest (재현에 필요한 설정·버전·데이터 해시·점수)")
print(out_core["SUBCLASS"].value_counts().head(10))


저장: ./submission_claude_v2_norm_core.csv
저장: ./submission_claude_v2_norm_dup.csv


저장: run_manifest (재현에 필요한 설정·버전·데이터 해시·점수)
SUBCLASS
COAD      556
STES      342
KIPAN     167
BRCA      135
OV        133
PRAD      113
KIRC       95
SKCM       88
LGG        86
GBMLGG     86
Name: count, dtype: int64


## 7. 버전 비교용 요약

In [21]:
# ---- 버전 비교용 요약 저장 ----
_f = lambda p: float(f1_score(y, p.argmax(1), average="macro"))
_final_te = te_dup if dup_applied else te_core
_pred = pd.Series(le.classes_[_final_te.argmax(1)]).value_counts()
summary = dict(
    version=CFG["VERSION"], tag=os.environ.get("RUN_TAG", ""),
    cfg={k: CFG[k] for k in ("AUG_NOISE", "GENE_MODE", "INNER_ES", "FULL_REFIT", "XGB_LR", "XGB_SEEDS", "USE_LGB",
                             "LR_CS", "USE_PAIR_BIAS", "PROFILE_WEIGHT")},
    per_model={k: _f(v) for k, v in RES["oof"].items()},
    per_model_noise=({k: _f(v) for k, v in RES["noise"].items()} if RES["noise"] is not None else None),
    weights=best_w, iters={k: [int(i) for i in v] for k, v in RES["iters"].items()}, lr_C=RES["lr_C"],
    oof_model=_f(oof_ens), oof_model_acc=float(accuracy_score(y, oof_ens.argmax(1))),
    oof_noise=(_f(noise_ens) if noise_ens is not None else None),
    oof_dup=float(f1), oof_final_nested=_f(FINAL_OOF),
    scale_applied=scale_applied, pair_applied=pair_applied, dup_applied=dup_applied,
    test_pred_top=_pred.head(8).to_dict(), test_coad=int(_pred.get("COAD", 0)),
    elapsed_cv_sec=float(time.time() - _T_CV),
)
os.makedirs("results", exist_ok=True)
_name = CFG["VERSION"] + (("_" + summary["tag"]) if summary["tag"] else "")
with open(f"results/{_name}.json", "w", encoding="utf-8") as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print(json.dumps(summary, ensure_ascii=False, indent=1, default=str))


{
 "version": "claude_v2",
 "tag": "",
 "cfg": {
  "AUG_NOISE": false,
  "GENE_MODE": "all",
  "INNER_ES": true,
  "FULL_REFIT": true,
  "XGB_LR": 0.08,
  "XGB_SEEDS": [
   42
  ],
  "USE_LGB": false,
  "LR_CS": [
   0.1
  ],
  "USE_PAIR_BIAS": false,
  "PROFILE_WEIGHT": false
 },
 "per_model": {
  "xgb": 0.45563069722521726,
  "lr": 0.42386129065794037
 },
 "per_model_noise": {
  "xgb": 0.23846748953787467,
  "lr": 0.3067092357536238
 },
 "weights": {
  "xgb": 0.3,
  "lr": 0.7
 },
 "iters": {
  "xgb": [
   240,
   201,
   256,
   212,
   230
  ]
 },
 "lr_C": 0.1,
 "oof_model": 0.4708520596485902,
 "oof_model_acc": 0.4515400741815836,
 "oof_noise": 0.3025093342873767,
 "oof_dup": 0.5367720146271897,
 "oof_final_nested": 0.5367720146271897,
 "scale_applied": false,
 "pair_applied": false,
 "dup_applied": true,
 "test_pred_top": {
  "COAD": 556,
  "STES": 342,
  "KIPAN": 178,
  "BRCA": 134,
  "OV": 132,
  "GBMLGG": 113,
  "PRAD": 108,
  "KIRC": 103
 },
 "test_coad": 556,
 "elapsed_cv_sec